In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/Colab Notebook/archive.zip" -d "/content/dataset"

ValueError: mount failed

Importing libraries and checking for right GPU. GPU can be selected as T4 GPU

In [ ]:
import os
import time
import shutil
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

import tensorflow as tf

from tensorflow.keras import (
    layers,
    models,
    optimizers,
    regularizers,
)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger,
)

warnings.filterwarnings("ignore")

print("TensorFlow Version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("[INFO] GPU is active")

    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

    tf.keras.mixed_precision.set_global_policy("mixed_float16")

else:
    print("GPU cannot found")

Parameters, Drive and dataset setup.
*Note: To successfully import dataset, update the DATASET_PATH as needed.*

In [ ]:
DATASET_PATH = "/content/dataset"
OUTPUT_DIR = "/content/cnn_outputs"

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/cnn_outputs"

EMOTION_LABELS = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise",
]

IMG_SIZE = 48
CHANNELS = 1

INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, CHANNELS)

NUM_CLASSES = len(EMOTION_LABELS)

# parameters

BATCH_SIZE = 32

EPOCHS = 25

LR_INITIAL = 1e-3
LR_MIN = 1e-6

PATIENCE_ES = 7
PATIENCE_LR = 3

LR_FACTOR = 0.5

SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

Data Augmentation:

In [ ]:
def make_generators(
    dataset_path: str,
    batch_size: int = BATCH_SIZE,
):

    train_dir = os.path.join(dataset_path, "train")
    test_dir = os.path.join(dataset_path, "test")

    if not os.path.exists(train_dir):
        raise FileNotFoundError(
            f"Train folder cannot found: {train_dir}"
        )

    if not os.path.exists(test_dir):
        raise FileNotFoundError(
            f"Test folder cannot found: {test_dir}"
        )

    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        rotation_range=15,
        width_shift_range=0.10,
        height_shift_range=0.10,
        horizontal_flip=True,
        zoom_range=0.15,
        brightness_range=[0.8, 1.2],
        fill_mode="nearest",
    )

    test_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0
    )

    common_kwargs = dict(
        target_size=(IMG_SIZE, IMG_SIZE),
        color_mode="grayscale",
        batch_size=batch_size,
        class_mode="categorical",
        classes=EMOTION_LABELS,
        seed=SEED,
    )

    train_gen = train_datagen.flow_from_directory(
        train_dir,
        shuffle=True,
        **common_kwargs
    )

    test_gen = test_datagen.flow_from_directory(
        test_dir,
        shuffle=False,
        **common_kwargs
    )

    print(f"Train ex: {train_gen.samples}")
    print(f"Test ex: {test_gen.samples}")

    print(f"Classes:")
    print(train_gen.class_indices)

    steps_train = int(np.ceil(train_gen.samples / batch_size))

    steps_test = int(np.ceil(test_gen.samples / batch_size))

    return (
        train_gen,
        test_gen,
        steps_train,
        steps_test,
    )

CNN Layers:

In [ ]:
def build_cnn(
    input_shape: tuple = INPUT_SHAPE,
    num_classes: int = NUM_CLASSES
):

    def conv_block(x, filters, dropout_rate):

        x = layers.Conv2D(
            filters,
            (3, 3),
            padding="same",
            use_bias=False,
            kernel_regularizer=regularizers.l2(1e-4),
        )(x)

        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)

        x = layers.Conv2D(
            filters,
            (3, 3),
            padding="same",
            use_bias=False,
            kernel_regularizer=regularizers.l2(1e-4),
        )(x)

        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(dropout_rate)(x)

        return x

    inp = layers.Input(
        shape=input_shape,
        name="input_face"
    )

    x = inp

    x = conv_block(x, 32, 0.20)
    x = conv_block(x, 64, 0.25)
    x = conv_block(x, 128, 0.30)
    x = conv_block(x, 256, 0.35)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        256,
        kernel_regularizer=regularizers.l2(1e-4),
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="predictions",
    )(x)

    model = models.Model(
        inputs=inp,
        outputs=outputs,
        name="FER_CNN_FAST",
    )

    return model


Compile:

In [ ]:
def compile_model(
    model,
    lr: float = LR_INITIAL
):

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=lr
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

Callbacks:

In [ ]:
def make_callbacks(
    output_dir: str = OUTPUT_DIR
):

    weights_path = os.path.join(
        output_dir,
        "best_model.keras"
    )

    cb = [

        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE_ES,
            restore_best_weights=True,
            verbose=1,
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=LR_FACTOR,
            patience=PATIENCE_LR,
            min_lr=LR_MIN,
            verbose=1,
        ),

        ModelCheckpoint(
            filepath=weights_path,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
        ),

        CSVLogger(
            os.path.join(
                output_dir,
                "training_log.csv"
            )
        ),
    ]

    return cb

Training:

In [ ]:
def train_model(
    model,
    train_gen,
    test_gen,
    steps_train,
    steps_test,
    epochs: int = EPOCHS,
):

    print(f"\n[INFO] Training starts: ({epochs} epoch)\n")

    t0 = time.time()

    history = model.fit(
        train_gen,
        steps_per_epoch=steps_train,
        epochs=epochs,
        validation_data=test_gen,
        validation_steps=steps_test,
        callbacks=make_callbacks(),
        verbose=1,
    )

    elapsed = time.time() - t0

    print(
        f"\n[INFO] Trainign finished: "
        f"{elapsed/60:.1f} min"
    )

    return history

In [ ]:
def evaluate(
    model,
    generator,
    split_name: str,
):

    generator.reset()

    probs = model.predict(
        generator,
        verbose=0
    )

    y_pred = np.argmax(
        probs,
        axis=1
    )

    y_true = generator.classes

    acc = accuracy_score(
        y_true,
        y_pred
    )

    print("\n" + "="*50)

    print(
        f"[{split_name}] Accuracy: "
        f"{acc*100:.2f}%"
    )

    print("="*50)

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=EMOTION_LABELS,
            digits=4,
        )
    )

    return y_true, y_pred, acc


def plot_confusion_matrix(
    y_true,
    y_pred,
):

    cm = confusion_matrix(
        y_true,
        y_pred,
        normalize="true",
    )

    plt.figure(figsize=(9, 7))

    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=EMOTION_LABELS,
        yticklabels=EMOTION_LABELS,
    )

    plt.title(
        "CNN Confusion Matrix",
        fontsize=14,
        fontweight="bold",
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")

    save_path = os.path.join(
        OUTPUT_DIR,
        "confusion_matrix.png"
    )

    plt.savefig(save_path, dpi=150)

    print(f"[INFO] Saved: {save_path}")

    plt.show()


def plot_training_history(history):

    h = history.history

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5)
    )

    # Accuracy
    axes[0].plot(
        h["accuracy"],
        label="Train Accuracy",
        linewidth=2,
    )

    axes[0].plot(
        h["val_accuracy"],
        label="Val Accuracy",
        linewidth=2,
        linestyle="--",
    )

    axes[0].set_title(
        "Accuracy vs Epoch"
    )

    axes[0].legend()

    axes[0].grid(alpha=0.3)

    # Loss
    axes[1].plot(
        h["loss"],
        label="Train Loss",
        linewidth=2,
    )

    axes[1].plot(
        h["val_loss"],
        label="Val Loss",
        linewidth=2,
        linestyle="--",
    )

    axes[1].set_title(
        "Loss vs Epoch"
    )

    axes[1].legend()

    axes[1].grid(alpha=0.3)

    plt.tight_layout()

    save_path = os.path.join(
        OUTPUT_DIR,
        "training_history.png"
    )

    plt.savefig(save_path, dpi=150)

    print(f"[INFO] sadev: {save_path}")

    plt.show()



Save and Main

In [ ]:
def save_model(model):

    save_path = os.path.join(
        OUTPUT_DIR,
        "final_model.keras"
    )

    model.save(save_path)

    print(f"Model is saved: {save_path}")

# main

print(f"\n Dataset: {DATASET_PATH}")

train_gen, test_gen, steps_train, steps_test = make_generators(
    DATASET_PATH,
    batch_size=BATCH_SIZE,
)

model = build_cnn()

compile_model(model)

model.summary()

history = train_model(
    model,
    train_gen,
    test_gen,
    steps_train,
    steps_test,
)

y_true, y_pred, test_acc = evaluate(
    model,
    test_gen,
    split_name="TEST",
)


plot_training_history(history)

plot_confusion_matrix(
    y_true,
    y_pred,
)


save_model(model)

if os.path.exists(DRIVE_OUTPUT_DIR):
    shutil.rmtree(DRIVE_OUTPUT_DIR)

shutil.copytree(
    OUTPUT_DIR,
    DRIVE_OUTPUT_DIR,
)

print(
    f"\n Saved to Drive:\n"
    f"{DRIVE_OUTPUT_DIR}"
)

print(
    f"\n[DONE] Test Accuracy: "
    f"{test_acc*100:.2f}%"
)